# 04 · Forecast scores, calibration and model risk

**RQ3:** Does the conclusion survive changing the loss and dependence assumptions?
The frozen benchmark has an informative ranking reversal. QLIKE remains primary;
the quadratic score is retained with paired uncertainty, rather than promoted
after seeing favorable outcomes.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from spde_covariance.empirical_research import load_research_evidence

evidence = load_research_evidence(Path("../experiments/results/shfe-research"))
print(evidence["protocol"]["status"])
print("Numerical source:", evidence["manifest"]["git_revision"])
print("Original baseline source:", evidence["protocol"]["baseline_source_revision"])
assert evidence["protocol"]["baseline_summary_sha256"]
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def table(rows, columns):
    print(" | ".join(columns))
    for row in rows:
        print(
            " | ".join(
                (
                    f"{row.get(c):.6g}" if isinstance(row.get(c), float) else str(row.get(c))
                    for c in columns
                )
            )
        )

## What the scores measure

For $z=Y^2$ and positive forecast $v$,

```math
\ell_Q(v,z)=\log v+z/v,\qquad \ell_2(v,z)=v^2-2vz.
```

Paired differences are affine in $z$. If $m=\mathbb E[z\mid\mathcal F]$, expected
regrets are $\log(v/m)+m/v-1$ and $(v-m)^2$. Both target the conditional second
moment but weight misspecification differently. QLIKE is sensitive to severe
relative underprediction; quadratic loss weights cash-scale errors. A Gaussian
distribution is not needed. QLIKE uses the original positive floor; quadratic
loss uses the unfloored nonnegative forecast. See
[Patton (2011)](https://public.econ.duke.edu/~ap172/Patton_vol_proxies_JoE_2011.pdf).

In [ ]:
scores = evidence["scores"]
print(json.dumps(scores["protocol"], indent=2))
paired = [
    r
    for r in scores["paired_comparisons"]
    if r["stratum"] == "pooled" and r["method"] == "spde_local"
]
table(
    paired,
    [
        "exposure",
        "loss",
        "common_count",
        "mean",
        "bootstrap_standard_error",
        "bootstrap_percentile_95",
    ],
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, loss, scale in zip(axes, ["qlike", "quadratic"], [1.0, 1000000000000.0], strict=True):
    rows = [r for r in paired if r["loss"] == loss and r["available"]]
    for i, r in enumerate(rows):
        lo, hi = np.asarray(r["bootstrap_percentile_95"]) / scale
        ax.plot([lo, hi], [i, i], color="black")
        ax.plot(r["mean"] / scale, i, "o", color="black")
    ax.axvline(0, color="0.5", linestyle=":")
    ax.set_yticks(range(len(rows)), [r["exposure"] for r in rows])
    ax.set(title=loss, xlabel="SPDE − direct" + (" (10¹² CNY⁴)" if scale > 1 else ""))
fig.tight_layout()
plt.show()

## Dependence, calibration and period sensitivity

Circular moving blocks resample chronologically ordered paired scored sessions.
The number of sessions is not a Monte Carlo replication count. The percentile
ranges are descriptive and do not correct for the many explored comparisons.
Calibration bins are based on forecast risk, and realized-outcome strata are
explicitly ex post. Neither is used to tune a policy on the evaluation outcomes.

In [ ]:
sensitivity = [
    r
    for r in scores["block_sensitivity"]
    if r["method"] == "spde_local" and r["exposure"] == "spread_3m_6m"
]
table(
    sensitivity,
    [
        "loss",
        "requested_block_length",
        "mean",
        "bootstrap_standard_error",
        "bootstrap_percentile_95",
    ],
)
table(
    [
        r
        for r in scores["method_summaries"]
        if r["method"] in ("spde_local", "direct") and r["stratum"] == "year"
    ],
    ["group", "exposure", "method", "count", "mean_qlike", "risk_ratio"],
)

In [ ]:
rows = [r for r in scores["calibration_bins"] if r["method"] == "spde_local"]
table(
    rows,
    [
        "exposure",
        "forecast_variance_quintile",
        "count",
        "predicted_rms_cny",
        "realized_rms_cny",
        "risk_ratio",
    ],
)
fig, ax = plt.subplots(figsize=(5, 4))
for exposure in sorted({r["exposure"] for r in rows}):
    selected = [r for r in rows if r["exposure"] == exposure and r["available"]]
    ax.plot(
        [r["predicted_rms_cny"] for r in selected],
        [r["realized_rms_cny"] for r in selected],
        "o-",
        label=exposure,
    )
limit = ax.get_xlim()[1]
ax.plot([0, limit], [0, limit], ":", color="0.5")
ax.set(
    xlabel="Forecast RMS (CNY)",
    ylabel="Realized RMS within bin (CNY)",
    title="Forecast-risk calibration bins",
)
ax.legend(fontsize=7)
fig.tight_layout()
plt.show()

## Influential paired differences

The following diagnostic removes the largest 1% or 5% of absolute paired score
differences. It measures influence, not a preferred cleaned evaluation sample.
No dates or individual market outcomes are exported; small cells are suppressed.

In [ ]:
table(
    [r for r in scores["score_influence"] if r["method"] == "spde_local"],
    [
        "exposure",
        "loss",
        "fraction",
        "full_mean_difference",
        "remaining_mean_difference",
        "removed_absolute_difference_share",
    ],
)
table(
    [r for r in scores["availability"] if r["method"] in ("direct", "spde_local")],
    list(scores["availability"][0].keys()),
)

## Interpretation and next decision

Read score ranking, calibration and uncertainty together. Favorable quadratic
loss on one exposure does not remove a QLIKE or underprediction failure. The
matched-window experiment addresses attribution; the joint model addresses
missing risk. An uninspected confirmation period is needed for a revised policy.

## Reproduction and research boundary

This notebook reads checksummed aggregate evidence, not raw exchange files.
The numerical algorithms are in `spde_covariance`; use the
[protocol](../docs/empirical-research-protocol.md) to rerun local calibration.
Execute this notebook with `uv run python scripts/check_notebook.py notebooks/04_forecast_scores_and_model_risk.ipynb`.

The original historical benchmark remains fixed. New analyses on the inspected
2024–2025 period are exploratory, even though forecast issuance is past-only.
No final untouched confirmation period, executable trading backtest or physical
forcing identification is claimed. See the [roadmap](../README.md).